In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
print(np.__version__)
print(pd.__version__)

2.0.2
2.2.2


In [3]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer
from sklearn.model_selection import train_test_split

In [2]:
df = pd.read_csv(r"cleaned_political_bias_data (1).csv")
df.sample()

,News_ID,Title,News_Body,Stance,Label,issue,topic,roundup
3532,1178,trump we need republican roy moore to win in a...,president trump on monday offered his most exp...,lean left,liberal,trump endorses roy moore,elections,president trump on monday offered his most exp...


## Configuration

In [9]:
import torch
import os

# ── Paths ──────────────────────────────────────────────────
OUTPUT_DIR      = '/content/model_outputs'
MODEL_SAVE_PATH = os.path.join(OUTPUT_DIR, 'best_model.pt')
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Model ──────────────────────────────────────────────────
MODEL_NAME  = 'bert-base-uncased'
MAX_LENGTH  = 256
NUM_CLASSES = 3

# ── Training ───────────────────────────────────────────────
BATCH_SIZE    = 16
LEARNING_RATE = 2e-5
NUM_EPOCHS    = 4
WARMUP_STEPS  = 500
WEIGHT_DECAY  = 0.01
DROPOUT_RATE  = 0.3
EARLY_STOPPING_PATIENCE = 2
GRADIENT_CLIP = 1.0

# ── Data Split ─────────────────────────────────────────────
TRAIN_SIZE   = 0.70
VAL_SIZE     = 0.15
TEST_SIZE    = 0.15
RANDOM_STATE = 42

# ── Labels ─────────────────────────────────────────────────
LABEL_MAP   = {'liberal': 0, 'conservative': 1, 'center': 2}
LABEL_NAMES = ['Liberal', 'Conservative', 'Center']

# ── Device ─────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Using device: {DEVICE}")
if DEVICE.type == 'cuda':
    print(f"   GPU: {torch.cuda.get_device_name(0)}")

✅ Using device: cpu


In [15]:
import pandas as pd

# Directly use the uploaded CSV
DATA_PATH = 'cleaned_political_bias_data (1).csv'

# Quick sanity check
df_check = pd.read_csv(DATA_PATH, nrows=3)
print(f"✅ File found!")
print(f"Columns: {df_check.columns.tolist()}")
print(f"Sample labels: {df_check['Label'].tolist()}")

✅ File found!
Columns: ['News_ID', 'Title', 'News_Body', 'Stance', 'Label', 'issue', 'topic', 'roundup']
Sample labels: ['liberal', 'conservative', 'center']


## Data Preparation

In [19]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer
from sklearn.model_selection import train_test_split


class NewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts      = texts
        self.labels     = labels
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        # ── Fix 1: use __call__ instead of encode_plus ──────
        encoding = self.tokenizer(
            str(self.texts[idx]),
            add_special_tokens   = True,
            max_length           = self.max_length,
            padding              = 'max_length',
            truncation           = True,
            return_attention_mask= True,
            return_tensors       = 'pt'
        )
        return {
            'input_ids'     : encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'label'         : torch.tensor(self.labels[idx], dtype=torch.long)
        }


def load_and_preprocess_data(data_path):
    print("=" * 60)
    print("LOADING AND PREPROCESSING DATA")
    print("=" * 60)

    df = pd.read_csv(data_path)
    print(f"Total samples loaded : {len(df)}")

    required = ['Title', 'News_Body', 'Label']
    before = len(df)
    df = df.dropna(subset=required)
    print(f"After dropping nulls : {len(df)}  (removed {before - len(df)})")

    df['Label'] = df['Label'].str.strip().str.lower()

    for col in ['Stance', 'issue', 'topic', 'roundup']:
        if col not in df.columns:
            df[col] = ''
        df[col] = df[col].fillna('')

    def build_train_text(row):
        parts = [row['Title']]
        if row['Stance']:  parts.append(f"Stance: {row['Stance']}")
        if row['issue']:   parts.append(f"Issue: {row['issue']}")
        if row['topic']:   parts.append(f"Topic: {row['topic']}")
        if row['roundup']: parts.append(row['roundup'])
        parts.append(row['News_Body'])
        return ' [SEP] '.join(parts)

    df['train_text'] = df.apply(build_train_text, axis=1)
    df['infer_text'] = df['Title'] + ' [SEP] ' + df['News_Body']

    df['label_encoded'] = df['Label'].map(LABEL_MAP)
    unmapped = df['label_encoded'].isnull().sum()
    if unmapped:
        print(f"⚠️  {unmapped} rows with unknown labels dropped.")
        df = df.dropna(subset=['label_encoded'])
    df['label_encoded'] = df['label_encoded'].astype(int)

    print("\nLabel distribution:")
    print(df['Label'].value_counts())

    return df['train_text'].values, df['label_encoded'].values, df


def create_data_splits(texts, labels):
    print("\n" + "=" * 60)
    print("CREATING DATA SPLITS")
    print("=" * 60)

    X_tv, X_test, y_tv, y_test = train_test_split(
        texts, labels, test_size=TEST_SIZE,
        random_state=RANDOM_STATE, stratify=labels
    )
    val_ratio = VAL_SIZE / (TRAIN_SIZE + VAL_SIZE)
    X_train, X_val, y_train, y_val = train_test_split(
        X_tv, y_tv, test_size=val_ratio,
        random_state=RANDOM_STATE, stratify=y_tv
    )

    print(f"Train : {len(X_train)}  |  Val : {len(X_val)}  |  Test : {len(X_test)}")
    return X_train, X_val, X_test, y_train, y_val, y_test


def create_data_loaders(X_train, X_val, X_test, y_train, y_val, y_test):
    print("\n" + "=" * 60)
    print("CREATING DATA LOADERS")
    print("=" * 60)

    tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)
    print(f"Tokenizer loaded: {MODEL_NAME}")

    train_ds = NewsDataset(X_train, y_train, tokenizer, MAX_LENGTH)
    val_ds   = NewsDataset(X_val,   y_val,   tokenizer, MAX_LENGTH)
    test_ds  = NewsDataset(X_test,  y_test,  tokenizer, MAX_LENGTH)

    # ── Fix 2: num_workers=0 avoids DataLoader worker crashes ──
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    print(f"Batches — Train: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}")
    return train_loader, val_loader, test_loader, tokenizer


# ── Run ────────────────────────────────────────────────────
texts, labels, df = load_and_preprocess_data(DATA_PATH)
X_train, X_val, X_test, y_train, y_val, y_test = create_data_splits(texts, labels)
train_loader, val_loader, test_loader, tokenizer = create_data_loaders(
    X_train, X_val, X_test, y_train, y_val, y_test
)

print("\n✅ Data preprocessing complete!")
print(f"\nSample TRAINING text (first 300 chars):")
print(texts[0][:300])

LOADING AND PREPROCESSING DATA
Total samples loaded : 9190
After dropping nulls : 9189  (removed 1)

Label distribution:
Label
liberal         3065
conservative    3062
center          3062
Name: count, dtype: int64

CREATING DATA SPLITS
Train : 6431  |  Val : 1379  |  Test : 1379

CREATING DATA LOADERS
Tokenizer loaded: bert-base-uncased
Batches — Train: 402 | Val: 87 | Test: 87

✅ Data preprocessing complete!

Sample TRAINING text (first 300 chars):
north carolina house race carries unlikely hangover from fraud scandal [SEP] Stance: lean left [SEP] Issue: illegal ballot harvesting prompted re do of north carolinas eth district election [SEP] Topic: voting rights and voter fraud [SEP] charlotte no the closely watched special us house election be


## Model Description

In [17]:
import torch.nn as nn
from transformers import BertModel


class BertNewsClassifier(nn.Module):
    """
    BERT-based 3-class news bias classifier.
    Architecture: BERT encoder → Dropout → Linear(768 → 3)
    """

    def __init__(self, num_classes=NUM_CLASSES, dropout=DROPOUT_RATE):
        super().__init__()
        self.bert       = BertModel.from_pretrained(MODEL_NAME)
        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        outputs      = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled       = outputs.pooler_output          # [CLS] representation
        pooled       = self.dropout(pooled)
        return self.classifier(pooled)


def count_parameters(model):
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


# Instantiate
model = BertNewsClassifier().to(DEVICE)
total, trainable = count_parameters(model)
print(f"Total parameters     : {total:,}")
print(f"Trainable parameters : {trainable:,}")
print(f"\n✅ Model ready on {DEVICE}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Total parameters     : 109,484,547
Trainable parameters : 109,484,547

✅ Model ready on cpu


## Model Training

In [ ]:
import time
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from tqdm.notebook import tqdm


# ── Helpers ────────────────────────────────────────────────
def evaluate_loop(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for batch in loader:
            ids   = batch['input_ids'].to(device)
            mask  = batch['attention_mask'].to(device)
            lbls  = batch['label'].to(device)
            logits = model(ids, mask)
            loss   = criterion(logits, lbls)
            total_loss += loss.item()
            preds  = logits.argmax(dim=1)
            correct += (preds == lbls).sum().item()
            total   += lbls.size(0)
    return total_loss / len(loader), correct / total


def train_model(model, train_loader, val_loader, device):
    print("\n" + "=" * 60)
    print("TRAINING")
    print("=" * 60)

    optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    total_steps = len(train_loader) * NUM_EPOCHS
    scheduler   = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps   = WARMUP_STEPS,
        num_training_steps = total_steps
    )
    criterion = nn.CrossEntropyLoss()

    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    best_val_acc   = 0.0
    patience_count = 0

    for epoch in range(NUM_EPOCHS):
        start = time.time()
        model.train()
        total_loss, correct, total = 0.0, 0, 0

        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}")
        for batch in pbar:
            ids   = batch['input_ids'].to(device)
            mask  = batch['attention_mask'].to(device)
            lbls  = batch['label'].to(device)

            optimizer.zero_grad()
            logits = model(ids, mask)
            loss   = criterion(logits, lbls)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP)
            optimizer.step()
            scheduler.step()

            total_loss += loss.item()
            preds   = logits.argmax(dim=1)
            correct += (preds == lbls).sum().item()
            total   += lbls.size(0)
            pbar.set_postfix({'loss': f"{loss.item():.4f}"})

        train_loss = total_loss / len(train_loader)
        train_acc  = correct / total
        val_loss, val_acc = evaluate_loop(model, val_loader, criterion, device)
        elapsed = (time.time() - start) / 60

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)

        print(f"\nEpoch {epoch+1}/{NUM_EPOCHS}  ({elapsed:.1f} min)")
        print(f"  Train  — loss: {train_loss:.4f}  acc: {train_acc:.4f}")
        print(f"  Val    — loss: {val_loss:.4f}  acc: {val_acc:.4f}")

        # Save best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save({
                'epoch'            : epoch,
                'model_state_dict' : model.state_dict(),
                'val_acc'          : val_acc,
                'val_loss'         : val_loss,
            }, MODEL_SAVE_PATH)
            print(f"  💾 Best model saved  (val_acc={val_acc:.4f})")
            patience_count = 0
        else:
            patience_count += 1
            print(f"  ⏳ No improvement ({patience_count}/{EARLY_STOPPING_PATIENCE})")
            if patience_count >= EARLY_STOPPING_PATIENCE:
                print("  🛑 Early stopping triggered.")
                break

    print(f"\n✅ Training complete. Best val acc: {best_val_acc:.4f}")
    return history, best_val_acc


history, best_val_acc = train_model(model, train_loader, val_loader, DEVICE)


TRAINING


Epoch 1/4:   0%|          | 0/402 [00:00<?, ?it/s]

## Testing

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, precision_recall_fscore_support
)

# ── Load best model ────────────────────────────────────────
checkpoint = torch.load(MODEL_SAVE_PATH)
model.load_state_dict(checkpoint['model_state_dict'])
print(f"Best model loaded from epoch {checkpoint['epoch']+1}  (val_acc={checkpoint['val_acc']:.4f})")


# ── Collect test predictions ───────────────────────────────
model.eval()
all_preds, all_labels, all_probs = [], [], []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Evaluating"):
        ids   = batch['input_ids'].to(DEVICE)
        mask  = batch['attention_mask'].to(DEVICE)
        lbls  = batch['label'].to(DEVICE)

        logits = model(ids, mask)
        probs  = torch.softmax(logits, dim=1)
        preds  = logits.argmax(dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(lbls.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs  = np.array(all_probs)


# ── Classification report ──────────────────────────────────
print("\n" + "=" * 60)
print("CLASSIFICATION REPORT")
print("=" * 60)
acc = accuracy_score(all_labels, all_preds)
print(f"\nOverall Test Accuracy: {acc:.4f} ({acc*100:.2f}%)\n")
print(classification_report(all_labels, all_preds, target_names=LABEL_NAMES, digits=4))


# ── Confusion matrix ───────────────────────────────────────
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES)
plt.title('Confusion Matrix', fontsize=14)
plt.ylabel('True Label'); plt.xlabel('Predicted Label')
plt.tight_layout()
cm_path = os.path.join(OUTPUT_DIR, 'confusion_matrix.png')
plt.savefig(cm_path, dpi=150)
plt.show()
print(f"Saved → {cm_path}")


# ── Training history ───────────────────────────────────────
epochs = range(1, len(history['train_loss']) + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(epochs, history['train_loss'], 'b-o', label='Train Loss')
ax1.plot(epochs, history['val_loss'],   'r-o', label='Val Loss')
ax1.set_title('Loss'); ax1.set_xlabel('Epoch'); ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(epochs, history['train_acc'], 'b-o', label='Train Acc')
ax2.plot(epochs, history['val_acc'],   'r-o', label='Val Acc')
ax2.set_title('Accuracy'); ax2.set_xlabel('Epoch'); ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
hist_path = os.path.join(OUTPUT_DIR, 'training_history.png')
plt.savefig(hist_path, dpi=150)
plt.show()
print(f"Saved → {hist_path}")